# Geometries along the CO2 dissociation path

`structure/Co_5H2O_CO2_rhf_optimized.xyz` is the HF/def2-TZVP optimized equilibrium geometry of [Co(H2O)5CO2]^3+ (singlet), with a Co–O(CO2) distance of 1.97 Å.
The other geometries are made by moving the three CO2 atoms rigidly along the vector from the centroid of the Co(H2O)5 fragment to the CO2 centroid. Each shift scales that vector by a factor `shift`:

| shift | -0.15 | 0.0 | 0.15 | 0.3 | 0.45 | 0.6 | 0.75 | 0.9 | 1.05 | 1.2 |
|---|---|---|---|---|---|---|---|---|---|---|
| Co–O(CO2) [Å] | 1.44 | 1.97 | 2.51 | 3.04 | 3.57 | 4.10 | 4.63 | 5.16 | 5.69 | 6.23 |

Running this notebook rewrites the files in `structure/` with identical content.
The 3D viewer needs `py3Dmol`.

In [ ]:
import py3Dmol


mol_xyz_path = "./structure/Co_5H2O_CO2_rhf_optimized.xyz"
with open(mol_xyz_path) as f:
    xyz = f.read()


def show_mol(xyz):
    view = py3Dmol.view(width=800, height=600, style='sphere')
    view.addModel(xyz, 'xyz')
    view.setStyle({'stick': {}, "sphere":{"scale":0.3}})
    view.zoomTo()
    view.show()
    
show_mol(xyz)

In [ ]:
import numpy as np


def move_co2_from_geom_center_of_mol(mol_xyz_path, movement_factor):
    # Load the molecule
    atoms = []
    coords = []

    with open(mol_xyz_path, "r") as f:
        lines = f.readlines()[2:]  # Skipping the header (assuming first two lines are header)
        for line in lines:
            parts = line.split()
            atoms.append(parts[0])
            coords.append(np.array([float(x) for x in parts[1:4]]))

    coords = np.array(coords)

    # 0-based indices of the CO2 atoms (C, O, O) in the xyz file
    co2_indices = [16, 17, 18]

    # Create a mask to exclude CO2 atoms from center of mass calculation
    mask = np.ones(len(atoms), dtype=bool)
    mask[co2_indices] = False

    # Select atoms that are not part of CO2
    non_co2_coords = coords[mask]

    # Compute center of mass of non-CO2 atoms
    masses = np.ones(len(non_co2_coords))  # Assuming all atoms have the same mass, otherwise load specific masses
    center_of_mass = np.sum(non_co2_coords * masses[:, None], axis=0) / np.sum(masses)


    translation_vector = (coords[co2_indices].mean(axis=0) - center_of_mass) * movement_factor

    # Apply the translation to the CO2 atoms to move them away from the center of mass
    new_co2_coords = coords[co2_indices] + translation_vector

    # Replace the original CO2 coordinates with the new ones
    coords[co2_indices] = new_co2_coords

    # Save the displaced geometry next to the input file
    new_xyz_filename = mol_xyz_path[:-4]+f"_{movement_factor}_shifted"+mol_xyz_path[-4:]
    with open(new_xyz_filename, "w") as f:
        f.write(f"{len(atoms)}\n")
        f.write("Modified molecule with CO2 moved away from center of mass\n")
        for atom, coord in zip(atoms, coords):
            f.write(f"{atom} {' '.join(map(str, coord))}\n")
    
    return new_xyz_filename


In [ ]:
# Regenerate the shifted geometries used in the paper (shift = movement_factor).
# Shift 0.0 is the optimized equilibrium geometry itself; the shipped file for 0.0 keeps the original formatting.
shift_list = ['-0.15', '0.15', '0.3', '0.45', '0.6', '0.75', '0.9', '1.05', '1.2']
for shift in shift_list:
    new_xyz_filename = move_co2_from_geom_center_of_mol(mol_xyz_path, float(shift))
    print(new_xyz_filename)

In [ ]:
show_mol(open('./structure/Co_5H2O_CO2_rhf_optimized_1.2_shifted.xyz').read())